In [30]:
# Step 0) Load the data from Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
import numpy as np
import pandas as pd

PROJECT_DIR = '/content/drive/MyDrive/ESE 5420 Final Project'
raw = pd.read_csv(f'{PROJECT_DIR}/NASDAQ100_Historical_Data.csv', parse_dates=['Date']).sort_values(['Ticker', 'Date'], ignore_index=True)
raw.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(514075, 8)

In [31]:
# Step 1a) Per-row diagnostics and a summary of the panel
qc = raw.copy()
g = qc.groupby('Ticker')
qc['ret_1d'] = g['Adj Close'].pct_change()
calendar = pd.Series(range(qc['Date'].nunique()), index=np.sort(qc['Date'].unique()))  # all market trading days
qc['missed_days'] = qc['Date'].map(calendar).groupby(qc['Ticker']).diff() - 1         # market days skipped since previous row
span = g['Date'].agg(first='min', last='max', rows='size')

summary = pd.Series({
    'rows': len(raw),
    'tickers': len(span),
    'dates': f"{raw['Date'].min().date()} .. {raw['Date'].max().date()}",
    'missing values': raw.isna().sum().sum(),
    'duplicate (Ticker, Date)': raw.duplicated(['Ticker', 'Date']).sum(),
    'tickers listed after Jan 2000': (span['first'] > '2000-01-31').sum(),
    'tickers ending before last date': (span['last'] < raw['Date'].max()).sum(),
    'tickers with < 200 rows': (span['rows'] < 200).sum(),
    '|1-day return| > 25%': (qc['ret_1d'].abs() > 0.25).sum(),
    'zero-change days %': round(100 * (qc['ret_1d'] == 0).mean(), 2),
    'rows after a missing market day': (qc['missed_days'] > 0).sum(),
})
summary

,0
rows,514075
tickers,100
dates,2000-01-03 .. 2026-02-18
missing values,0
"duplicate (Ticker, Date)",0
tickers listed after Jan 2000,45
tickers ending before last date,0
tickers with < 200 rows,0
|1-day return| > 25%,201
zero-change days %,3.02


In [32]:
# Step 1b) Price and volume validity (prices are rounded to cents: 1-cent tolerance)
P, tol = ['Open', 'High', 'Low', 'Close', 'Adj Close'], 0.01
checks = {
    'price <= 0': (qc[P] <= 0).any(axis=1),
    'volume == 0': qc['Volume'] == 0,
    'Low > High': qc['Low'] > qc['High'] + tol,
    'Open outside [Low, High]': ~qc['Open'].between(qc['Low'] - tol, qc['High'] + tol),
    'Close outside [Low, High]': ~qc['Close'].between(qc['Low'] - tol, qc['High'] + tol),
    'Adj Close > Close': qc['Adj Close'] > qc['Close'] + tol,
    'Adj Close < $1': qc['Adj Close'] < 1,
    'Adj Close < $5': qc['Adj Close'] < 5,
}
validity = pd.DataFrame({'rows': {k: m.sum() for k, m in checks.items()},
                         'tickers': {k: qc.loc[m, 'Ticker'].nunique() for k, m in checks.items()}})
validity['share_%'] = (100 * validity['rows'] / len(qc)).round(3)
validity

,rows,tickers,share_%
price <= 0,0,0,0.000
volume == 0,1594,13,0.310
Low > High,0,0,0.000
"Open outside [Low, High]",0,0,0.000
"Close outside [Low, High]",0,0,0.000
Adj Close > Close,0,0,0.000
Adj Close < $1,12433,16,2.419
Adj Close < $5,54377,43,10.578


In [33]:
# Step 1c) Extreme 1-day moves (possible split errors)
extreme = qc.loc[qc['ret_1d'].abs() > 0.5, ['Ticker', 'Date', 'Close', 'Adj Close', 'ret_1d']].sort_values('ret_1d')
extreme

,Ticker,Date,Close,Adj Close,ret_1d
422746,REGN,2003-03-31,7.52,7.48,-0.565369
188,AAPL,2000-09-29,0.46,0.39,-0.512500
51400,AMD,2016-04-22,3.99,3.99,0.522901
475188,TTWO,2008-02-25,26.89,26.89,0.548963
493505,VRTX,2012-05-07,58.12,58.12,0.553595
373670,ON,2002-01-03,3.50,3.50,0.555556
403783,PDD,2022-03-16,42.62,42.62,0.560601
493743,VRTX,2013-04-19,85.60,85.60,0.619066
421969,REGN,2000-02-23,36.62,36.42,0.754335


In [34]:
# Step 1d) Market trading days a ticker is missing inside its own history
gaps = qc.loc[qc['missed_days'] > 0, ['Ticker', 'Date', 'missed_days']].sort_values('missed_days', ascending=False)
gaps

,Ticker,Date,missed_days


In [35]:
# Step 1e) The panel year by year
by_year = qc.groupby(qc['Date'].dt.year).agg(
    trading_days=('Date', 'nunique'), tickers=('Ticker', 'nunique'), rows=('Ticker', 'size'),
    median_adj_close=('Adj Close', 'median'),
    below_5_pct=('Adj Close', lambda s: 100 * (s < 5).mean()),
    zero_change_pct=('ret_1d', lambda r: 100 * (r == 0).mean()),
).round(1)
by_year

,trading_days,tickers,rows,median_adj_close,below_5_pct,zero_change_pct
Date,,,,,,
2000,252,58,14293,13.1,28.4,6.3
2001,248,59,14520,13.0,30.2,6.5
2002,252,60,15022,10.4,31.7,7.5
2003,252,60,15120,10.0,27.5,8.6
2004,252,62,15308,12.4,23.0,6.5
2005,252,64,15929,13.3,21.0,5.8
2006,251,64,16064,15.1,17.7,4.6
2007,251,68,16642,17.2,17.9,4.0
2008,253,69,17370,13.9,22.2,3.0


In [36]:
# Step 1f) Extreme events: single-stock jumps vs market-wide days
ext = qc.loc[qc['ret_1d'].abs() > 0.25]
rets = qc.pivot(index='Date', columns='Ticker', values='ret_1d')
market = pd.DataFrame({'mkt_ret': rets.mean(axis=1),                                     # equal-weighted market return
                       'up_share': (rets > 0).sum(axis=1) / rets.notna().sum(axis=1)})   # share of stocks that rose
market_days = pd.concat([market.nsmallest(8, 'mkt_ret'), market.nlargest(8, 'mkt_ret')]).round(3)
yearly = pd.DataFrame({
    'jumps_over_25pct': ext.groupby(ext['Date'].dt.year).size(),
    'avg_pairwise_corr': rets.groupby(rets.index.year).apply(lambda r: np.nanmean(r.corr().values[np.triu_indices(r.shape[1], 1)])),
    'one_sided_days_pct': market.groupby(market.index.year)['up_share'].apply(lambda s: 100 * ((s >= .8) | (s <= .2)).mean()),
    'mkt_vol_ann_pct': market.groupby(market.index.year)['mkt_ret'].std() * 100 * 252 ** .5,
}).fillna({'jumps_over_25pct': 0}).round(2)
display(ext['Ticker'].value_counts().head(10).rename('jumps_over_25pct'), market_days)
yearly

,jumps_over_25pct
Ticker,
BKNG,18
NVDA,11
SMCI,11
TTD,11
MNST,11
REGN,9
BIIB,8
AMZN,8
NFLX,8


,mkt_ret,up_share
Date,,
2020-03-16,-0.121,0.011
2020-03-12,-0.093,0.000
2008-10-15,-0.090,0.029
2008-12-01,-0.089,0.000
2001-09-17,-0.081,0.017
2020-03-09,-0.080,0.022
2008-09-29,-0.077,0.000
2000-04-14,-0.067,0.018
2008-10-13,0.112,0.986


,jumps_over_25pct,avg_pairwise_corr,one_sided_days_pct,mkt_vol_ann_pct
Date,,,,
2000,24.0,0.15,8.33,33.57
2001,29.0,0.24,17.74,33.78
2002,14.0,0.31,32.14,33.62
2003,10.0,0.28,29.76,21.88
2004,5.0,0.24,28.17,17.62
2005,6.0,0.22,20.63,14.51
2006,1.0,0.24,21.51,15.55
2007,2.0,0.28,27.89,17.32
2008,16.0,0.48,40.32,41.14


In [37]:
# Step 1g) Save the step-1 tables to Drive
out = f'{PROJECT_DIR}/outputs/step1'
os.makedirs(out, exist_ok=True)
for name, table in {'summary': summary, 'ticker_span': span, 'validity_checks': validity, 'extreme_returns': extreme,
                    'gaps': gaps, 'by_year': by_year, 'market_days': market_days, 'yearly_market': yearly}.items():
    table.to_csv(f'{out}/{name}.csv')

In [44]:
# Step 2-3) Label and 23 features (Kaggle's 27 minus 4 exact copies of the returns) on one price basis: split- AND dividend-adjusted
FEATURES = ['return_1d', 'return_5d', 'return_10d', 'return_20d',
            'sma_ratio_5', 'sma_ratio_10', 'sma_ratio_20', 'sma_ratio_50', 'sma_ratio_200',
            'ema_ratio_12', 'ema_ratio_26', 'macd_hist', 'rsi_14', 'bb_position',
            'volatility_10d', 'volatility_20d', 'volatility_60d', 'volume_sma_ratio_10', 'volume_sma_ratio_20',
            'daily_range', 'avg_range_10d', 'high_low_ratio', 'atr_14']

def make_features(d):
    k = d['Adj Close'] / d['Close']                          # that day's adjustment factor
    c, h, l = d['Adj Close'], d['High'] * k, d['Low'] * k    # adjusted close, high, low
    v = d['Volume'].where(d['Volume'] > 0)                   # zero volume -> missing
    r = c.pct_change()
    x = pd.DataFrame(index=d.index)
    for n in (1, 5, 10, 20):
        x[f'return_{n}d'] = c.pct_change(n)
    for n in (5, 10, 20, 50, 200):
        x[f'sma_ratio_{n}'] = c / c.rolling(n).mean()
    ema12, ema26 = (c.ewm(span=s, adjust=False, min_periods=s).mean() for s in (12, 26))
    x['ema_ratio_12'], x['ema_ratio_26'] = c / ema12, c / ema26
    macd = ema12 - ema26
    x['macd_hist'] = (macd - macd.ewm(span=9, adjust=False, min_periods=9).mean()) / c
    gain, loss = (s.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean() for s in (c.diff().clip(lower=0), -c.diff().clip(upper=0)))
    x['rsi_14'] = (100 * gain / (gain + loss)).mask(gain + loss == 0, 50)            # Wilder RSI; flat window -> 50
    x['bb_position'] = (c - c.rolling(20).mean()) / (2 * c.rolling(20).std())
    for n in (10, 20, 60):
        x[f'volatility_{n}d'] = r.rolling(n).std() * 252 ** 0.5
    for n in (10, 20):
        x[f'volume_sma_ratio_{n}'] = v / v.rolling(n, min_periods=n // 2).mean()
    x['daily_range'] = (h - l) / c
    x['avg_range_10d'] = x['daily_range'].rolling(10).mean()
    x['high_low_ratio'] = h / l
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    x['atr_14'] = tr.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean() / c       # Wilder ATR / price
    x['target'] = (c.shift(-1) > c).astype(float).where(c.shift(-1).notna())          # 1 if next day's Adj Close is higher
    return x.replace([np.inf, -np.inf], np.nan)

feats = pd.concat([make_features(d) for _, d in raw.groupby('Ticker')])
data = raw[['Ticker', 'Date']].join(feats).dropna()       # drops the 200-day warm-up, each ticker's last day, zero-volume days
data.shape

(491772, 26)

In [50]:
# Step 3 check) Perturbation test (instructor): scramble every price and volume AFTER day t; no feature on day t
# or earlier may change. Three random days t for every ticker. Then row accounting and feature ranges.
rng = np.random.default_rng(0)
P = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
n_tests = flips = 0
for _, d in raw.groupby('Ticker'):
    d = d.astype({'Volume': float})
    base = make_features(d)
    for i in rng.integers(199, len(d) - 1, size=3):                       # day t = row i of this ticker, after the warm-up
        after = np.arange(len(d)) > i
        noisy = d.copy()
        noisy.loc[after, P] *= rng.uniform(0.5, 1.5, size=(after.sum(), len(P)))
        pert = make_features(noisy)
        pd.testing.assert_frame_equal(base.loc[~after, FEATURES], pert.loc[~after, FEATURES], check_exact=True)
        assert not base.loc[after, FEATURES].equals(pert.loc[after, FEATURES])  # control: the scramble reached the code
        flips += int(base['target'].iloc[i] != pert['target'].iloc[i])         # day t's label uses day t+1, so it may flip
        n_tests += 1

warm, last = raw.groupby('Ticker').cumcount() < 199, raw.groupby('Ticker').cumcount(ascending=False) == 0
missing = feats[FEATURES + ['target']].isna().any(axis=1)
display(pd.Series({
    'perturbation tests passed (ticker, day t)': n_tests,
    'of which day-t label flipped': flips,
    'rows in raw file': len(raw),
    'dropped: 200-day warm-up': int(warm.sum()),
    "dropped: each ticker's last day": int(last.sum()),
    'dropped: other missing (zero volume etc.)': int((missing & ~warm & ~last).sum()),
    'rows in model table': len(data),
    'positive %': round(100 * data['target'].mean(), 2),
    'first date': data['Date'].min().date(),
}).rename('step 2-3'))
data[FEATURES].describe(percentiles=[.01, .5, .99]).T.round(3)

,step 2-3
"perturbation tests passed (ticker, day t)",300
of which day-t label flipped,147
rows in raw file,514075
dropped: 200-day warm-up,19900
dropped: each ticker's last day,100
dropped: other missing (zero volume etc.),2303
rows in model table,491772
positive %,50.58
first date,2000-10-16


,count,mean,std,min,1%,50%,99%,max
return_1d,491772.0,0.001,0.026,-0.565,-0.069,0.000,0.076,0.619
return_5d,491772.0,0.004,0.056,-0.668,-0.148,0.004,0.167,1.241
return_10d,491772.0,0.008,0.077,-0.745,-0.200,0.007,0.235,1.666
return_20d,491772.0,0.017,0.109,-0.803,-0.271,0.015,0.340,2.269
sma_ratio_5,491772.0,1.001,0.027,0.449,0.922,1.002,1.078,1.473
sma_ratio_10,491772.0,1.003,0.041,0.402,0.883,1.004,1.117,1.637
sma_ratio_20,491772.0,1.006,0.060,0.324,0.834,1.007,1.169,1.986
sma_ratio_50,491772.0,1.016,0.095,0.229,0.742,1.018,1.275,2.377
sma_ratio_200,491772.0,1.064,0.196,0.035,0.565,1.063,1.652,3.462
ema_ratio_12,491772.0,1.003,0.039,0.431,0.888,1.004,1.105,1.587


In [46]:
# Step 3 save) One model table for the whole group
data.to_parquet(f'{PROJECT_DIR}/outputs/model_table.parquet', index=False)

In [47]:
# Step 4) Train/test split by date: 12 annual tests (2012-2023) + one final test (2024-2026, opened once at the end)
days = pd.DatetimeIndex(data['Date'].drop_duplicates().sort_values())   # all trading days in the model table
TESTS = {Y: (f'{Y}-01-01', f'{Y}-12-31') for Y in range(2012, 2024)} | {'final': ('2024-01-01', '2026-12-31')}

def split(test):
    """Train = all days before the test period except the last one, whose label is the first test day's close (purge)."""
    start, end = map(pd.Timestamp, TESTS[test])
    train_end = days[days < start][-2]
    return data[data['Date'] <= train_end], data[data['Date'].between(start, end)]

rows = {}
for test in TESTS:
    tr, te = split(test)
    gap = days.searchsorted(te['Date'].min()) - days.searchsorted(tr['Date'].max())
    assert gap == 2, test                                     # exactly one purged trading day between train and test
    rows[test] = {'train': f"{tr['Date'].min().date()} .. {tr['Date'].max().date()}",
                  'test': f"{te['Date'].min().date()} .. {te['Date'].max().date()}",
                  'train_rows': len(tr), 'test_rows': len(te), 'test_days': te['Date'].nunique(),
                  'train_up_%': round(100 * tr['target'].mean(), 2),
                  'test_up_%': round(100 * te['target'].mean(), 2) if test != 'final' else 'sealed'}
split_table = pd.DataFrame(rows).T
split_table.to_csv(f'{PROJECT_DIR}/outputs/step4_splits.csv')
split_table

,train,test,train_rows,test_rows,test_days,train_up_%,test_up_%
2012,2000-10-16 .. 2011-12-29,2012-01-03 .. 2012-12-31,179932,18791,250,48.11,49.08
2013,2000-10-16 .. 2012-12-28,2013-01-02 .. 2013-12-31,198723,19567,252,48.19,51.77
2014,2000-10-16 .. 2013-12-30,2014-01-02 .. 2014-12-31,218285,20245,252,48.54,51.81
2015,2000-10-16 .. 2014-12-30,2015-01-02 .. 2015-12-31,238530,20216,252,48.81,50.14
2016,2000-10-16 .. 2015-12-30,2016-01-04 .. 2016-12-30,258745,20735,252,48.92,52.11
2017,2000-10-16 .. 2016-12-29,2017-01-03 .. 2017-12-29,279478,21031,251,49.14,54.03
2018,2000-10-16 .. 2017-12-28,2018-01-02 .. 2018-12-31,300508,21234,251,49.48,51.89
2019,2000-10-16 .. 2018-12-28,2019-01-02 .. 2019-12-31,321739,22041,252,49.65,54.9
2020,2000-10-16 .. 2019-12-30,2020-01-02 .. 2020-12-31,343778,22806,253,49.98,53.56
2021,2000-10-16 .. 2020-12-30,2021-01-04 .. 2021-12-31,366582,23187,252,50.22,53.21


In [48]:
# Step 5) Inner validation folds inside every training set, plus fixed day-bootstrap resamples of every test period.
# Noise control (instructor feedback): one fixed seed; every model and setting uses the SAME folds and the SAME
# bootstrap resamples, so comparisons are paired and the noisy test-year AUC cancels when two schemes are compared.
K, SEED, B = 5, 5420, 1000
SCHEMES = ('shuffled', 'date_grouped', 'walk_forward')
rng = np.random.default_rng(SEED)
data['fold_shuffled'] = rng.permutation(len(data)) % K                                       # rows at random
data['fold_date'] = data['Date'].map(pd.Series(rng.permutation(len(days)) % K, index=days))  # whole trading days at random

def inner_folds(test, scheme):
    """Yield (train_index, validation_index) inside the training set of `test`."""
    dev = split(test)[0]
    if scheme == 'walk_forward':                                   # each of the last 5 training years validates once
        last = dev['Date'].dt.year.max()
        for year in range(last - K + 1, last + 1):
            train_end = days[days < pd.Timestamp(f'{year}-01-01')][-2]      # purge the day before the validation year
            yield dev.index[dev['Date'] <= train_end], dev.index[dev['Date'].dt.year == year]
    else:
        fold = dev['fold_shuffled' if scheme == 'shuffled' else 'fold_date']
        for k in range(K):
            yield dev.index[fold != k], dev.index[fold == k]

# 1,000 resamples of whole trading days for each test period: counts[b, d] = times day d is drawn in resample b
BOOT = {}
for test, (start, end) in TESTS.items():
    n = ((days >= start) & (days <= end)).sum()
    BOOT[test] = rng.multinomial(n, np.full(n, 1 / n), size=B).astype(np.int16)
data[['Ticker', 'Date', 'fold_shuffled', 'fold_date']].to_parquet(f'{PROJECT_DIR}/outputs/step5_folds.parquet', index=False)

In [49]:
# Step 5 check) Folds are disjoint and cover the training set; same-day overlap exists only where it should
rows = []
for test in TESTS:
    dev_rows = len(split(test)[0])
    for scheme in SCHEMES:
        n_val = 0
        for k, (tr, va) in enumerate(inner_folds(test, scheme)):
            tr_dates, va_dates = data.loc[tr, 'Date'], data.loc[va, 'Date']
            same_day = 100 * va_dates.isin(set(tr_dates)).mean()                 # % of validation rows whose date is also in training
            assert tr.intersection(va).empty
            assert scheme == 'shuffled' or same_day == 0
            if scheme == 'walk_forward':
                assert days.searchsorted(va_dates.min()) - days.searchsorted(tr_dates.max()) == 2   # one purged day
            n_val += len(va)
            rows.append({'test': test, 'scheme': scheme, 'fold': k, 'train_rows': len(tr), 'val_rows': len(va),
                         'val_dates': f'{va_dates.min().date()} .. {va_dates.max().date()}', 'same_day_%': round(same_day, 1)})
        assert scheme == 'walk_forward' or n_val == dev_rows                     # random folds cover the whole training set
fold_table = pd.DataFrame(rows)
fold_table.to_csv(f'{PROJECT_DIR}/outputs/step5_folds_summary.csv', index=False)
fold_table[fold_table['test'] == 2018]

,test,scheme,fold,train_rows,val_rows,val_dates,same_day_%
90,2018,shuffled,0,240512,59996,2000-10-16 .. 2017-12-28,100.0
91,2018,shuffled,1,240434,60074,2000-10-16 .. 2017-12-28,100.0
92,2018,shuffled,2,240435,60073,2000-10-16 .. 2017-12-28,100.0
93,2018,shuffled,3,240207,60301,2000-10-16 .. 2017-12-28,100.0
94,2018,shuffled,4,240444,60064,2000-10-16 .. 2017-12-28,100.0
95,2018,date_grouped,0,240745,59763,2000-10-23 .. 2017-12-26,0.0
96,2018,date_grouped,1,241421,59087,2000-10-18 .. 2017-12-28,0.0
97,2018,date_grouped,2,240336,60172,2000-10-16 .. 2017-12-12,0.0
98,2018,date_grouped,3,238999,61509,2000-10-24 .. 2017-12-27,0.0
99,2018,date_grouped,4,240531,59977,2000-10-17 .. 2017-12-22,0.0


In [51]:
# Step 6) Frozen configuration: fixed BEFORE any model is trained and not edited after results are seen
import json
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

CONFIG = {
    'features': FEATURES, 'tests': [str(t) for t in TESTS], 'schemes': SCHEMES, 'folds': K, 'seed': SEED, 'bootstrap': B,
    'logistic_grid_C': [0.001, 0.01, 0.1, 1, 10],      # L2 penalty; selection analysis
    'lightgbm_grid_num_leaves': [8, 16, 32, 64, 128],   # same budget: 5 settings
    'logistic_fixed_C': 1,                              # bias analysis
    'lightgbm_fixed_num_leaves': 32,                    # bias analysis
    'lightgbm_other': {'n_estimators': 300, 'learning_rate': 0.05, 'min_child_samples': 200, 'subsample': 0.8,
                       'subsample_freq': 1, 'colsample_bytree': 0.8, 'random_state': SEED, 'n_jobs': -1, 'verbose': -1},
    'clip_quantiles': [0.01, 0.99],                     # learned on the training fold only
}
with open(f'{PROJECT_DIR}/outputs/frozen_config.json', 'w') as f:
    json.dump(CONFIG, f, indent=1)

class Clip(BaseEstimator, TransformerMixin):
    """Clip each feature to its 1st-99th percentile, learned on the training fold only."""
    def fit(self, X, y=None):
        self.lo_, self.hi_ = np.quantile(X, CONFIG['clip_quantiles'], axis=0)
        return self
    def transform(self, X):
        return np.clip(X, self.lo_, self.hi_)

def logistic(C):
    return make_pipeline(Clip(), StandardScaler(), LogisticRegression(C=C, max_iter=1000))   # every step fit inside the fold

def lightgbm(num_leaves):
    return LGBMClassifier(num_leaves=num_leaves, **CONFIG['lightgbm_other'])                # trees need no scaling

def day_bootstrap_auc(y, score, day_pos, counts, chunk=50):
    """AUC on every day-bootstrap resample: each row weighted by how often its trading day was drawn."""
    o = np.argsort(score, kind='mergesort')
    y, day_pos = y[o], day_pos[o]
    out = []
    for c in np.array_split(counts, max(1, len(counts) // chunk)):
        w = c[:, day_pos].astype(float)
        neg, pos = w * (1 - y), w * y
        out.append((pos * (np.cumsum(neg, axis=1) - neg)).sum(1) / (pos.sum(1) * neg.sum(1)))
    return np.concatenate(out)

In [52]:
# Step 7) Baseline for one test year: logistic regression (C = 1) under the three schemes vs the year-Y AUC
def cv_auc(test, scheme, make_model):
    """CV estimate = mean validation AUC over the 5 inner folds of the test's training set."""
    aucs = []
    for tr, va in inner_folds(test, scheme):
        m = make_model().fit(data.loc[tr, FEATURES].to_numpy(), data.loc[tr, 'target'].to_numpy())
        aucs.append(roc_auc_score(data.loc[va, 'target'], m.predict_proba(data.loc[va, FEATURES].to_numpy())[:, 1]))
    return np.mean(aucs)

def test_auc(test, make_model):
    """Year-Y AUC of a model trained on the whole training set, with its 95% day-bootstrap interval."""
    tr, te = split(test)
    p = make_model().fit(tr[FEATURES].to_numpy(), tr['target'].to_numpy()).predict_proba(te[FEATURES].to_numpy())[:, 1]
    y = te['target'].to_numpy()
    start, end = TESTS[test]
    day_pos = days[(days >= start) & (days <= end)].get_indexer(te['Date'])
    assert np.isclose(day_bootstrap_auc(y, p, day_pos, np.ones((1, day_pos.max() + 1))), roc_auc_score(y, p))
    return roc_auc_score(y, p), *np.percentile(day_bootstrap_auc(y, p, day_pos, BOOT[test]), [2.5, 97.5])

YEAR = 2018
make = lambda: logistic(CONFIG['logistic_fixed_C'])
auc, lo, hi = test_auc(YEAR, make)
one_year = pd.DataFrame({s: {'cv_auc': cv_auc(YEAR, s, make)} for s in SCHEMES}).T.assign(test_auc=auc, test_lo=lo, test_hi=hi)
one_year['error'] = one_year['cv_auc'] - one_year['test_auc']
one_year.round(4)

,cv_auc,test_auc,test_lo,test_hi,error
shuffled,0.5264,0.5157,0.4934,0.5395,0.0107
date_grouped,0.5233,0.5157,0.4934,0.5395,0.0076
walk_forward,0.5187,0.5157,0.4934,0.5395,0.0030
